# Ejemplo: un agente por clase con `ninaspro`

Recorre el curso **IA Aplicada: Agentes** clase por clase. Cada sección agrega lo que se enseñó ese día.

**Antes de empezar:** Archivo → Guardar una copia en Drive. Después ejecuta la celda de preparación.

In [ ]:
#@title Preparación (ejecútala primero)
!pip install -q git+https://github.com/nifalconi/ninaspro
%env OPENROUTER_API_KEY=pega-aqui-la-key-del-curso

## Clase 2 · Primer agente
Solo las instrucciones son obligatorias. Lo demás se escribe con su nombre.

In [ ]:
from ninaspro import crear_agente, herramienta, interfaz

tutor = crear_agente("Ayudar con tareas de matemáticas", nombre="Mati")
tutor.chat("¿Cómo te llamas?")

## Clase 3 · Ficha de identidad
`ver_instrucciones()` muestra el texto que se arma con la ficha y que lee el modelo.

In [ ]:
tutor = crear_agente(
    "Ayudar a estudiantes de secundaria con sus tareas.",
    nombre="Tutor Socrático",
    tono="Cercano y paciente. No das la respuesta: guías con preguntas.",
    audiencia="Estudiantes de 14 a 18 años.",
    limitaciones=["Solo temas escolares.", "No des la respuesta final."],
)
tutor.ver_instrucciones()

## Clase 4 · Herramientas
El agente lee el docstring para saber para qué sirve la herramienta. `ver_pasos=True` muestra cuándo la usa.

In [ ]:
@herramienta
def calcular_tiempo(horas_libres: int, materias: int) -> str:
    """Reparte las horas libres de la semana entre las materias."""
    return f"{horas_libres / materias:.1f} horas por materia"

tutor = crear_agente("Ayudar a organizar el estudio.", herramientas=[calcular_tiempo])
tutor.chat("Tengo 6 horas y 3 materias", ver_pasos=True)

## Clase 5 · Habilidades
Cada habilidad es una carpeta dentro de `SKILLS/` con un `SKILL.md`. La primera celda crea una de ejemplo; en el curso ya vienen hechas.

In [ ]:
import os
os.makedirs("SKILLS/tutor-socratico", exist_ok=True)
with open("SKILLS/tutor-socratico/SKILL.md", "w", encoding="utf-8") as f:
    f.write("""---
name: tutor-socratico
description: Guiar con preguntas, sin dar la respuesta final.
---

Antes de explicar, pregunta qué sabe la estudiante del tema.
Da una pista por vez y termina siempre con una pregunta.
""")

In [ ]:
tutor = crear_agente("Ayudar con tareas.", habilidades=["tutor-socratico"])
tutor.chat("¿Cuánto es 3/4 + 1/2?", ver_pasos=True)

## Clase 6 · Memoria
Sin `memoria=True`, el agente olvida todo entre un mensaje y otro.

In [ ]:
tutor = crear_agente("Ayudar con tareas.", memoria=True)
tutor.chat("Me llamo Antonia")
tutor.chat("¿Cómo me llamo?")
tutor.borrar_memoria()

## Clase 7 · Guardrails
`limitaciones` es texto que el modelo puede saltarse. Un guardrail es código: bloquea siempre.

In [ ]:
def validar_entrada(mensaje):
    if "hackear" in mensaje.lower():
        return False, "Eso está fuera de lo que puedo ayudarte."
    return True, ""

tutor = crear_agente("Ayudar con tareas.", guardrails_entrada=validar_entrada)
tutor.chat("¿Cómo puedo hackear el wifi?", ver_pasos=True)

## Clase 9 · Interfaz
Cada persona que abre el link tiene su propia conversación.

In [ ]:
tutor = crear_agente("Ayudar con tareas.", nombre="Tutor", memoria=True)
interfaz(tutor, ejemplos=["¿Cómo sumo fracciones?"], mensaje_inicial="¡Hola! ¿En qué tarea estás?")

## Opcional · Dos agentes
**Uno usa al otro como herramienta.**

In [ ]:
traductor = crear_agente("Traducir al inglés. Devuelve solo la traducción.", nombre="Traductor",
                         especialidad="Traduce un texto al inglés")
profe = crear_agente("Enseñar inglés. Para traducir usa al Traductor.", herramientas=[traductor])
profe.chat("¿Cómo se dice 'me gusta programar'?", ver_pasos=True)

**Uno le pasa la conversación al otro.**

In [ ]:
experta = crear_agente("Orientar sobre becas para estudiar en EE.UU.", nombre="Experta en becas",
                       especialidad="Responde preguntas sobre becas")
recepcion = crear_agente("Derivar a quien corresponda.", nombre="Recepción", derivar_a=[experta])
recepcion.chat("¿Qué becas hay para estudiar en EE.UU.?", ver_pasos=True)

**Uno revisa lo que le escriben al otro** (guardrail semántico).

In [ ]:
clasificador = crear_agente("Responder SOLO 'VÁLIDO' si es sobre estudios; si no, 'INVÁLIDO'.")

def validar_semantico(mensaje):
    if "INVÁLIDO" in clasificador.obtener_respuesta(mensaje):
        return False, "Solo puedo ayudarte con temas de estudio."
    return True, ""

tutor = crear_agente("Ayudar con tareas escolares.", guardrails_entrada=validar_semantico)
tutor.chat("Dame una receta de pizza", ver_pasos=True)